# Data Cleaning & Transformation

This notebook focuses on cleaning and transforming the raw datasets used in the Canadian City Intelligence project.

The main objectives are to:

- Verify and standardize city names.
- Convert relevant columns to appropriate numeric and date data types.
- Reassess missing values after data types and formats have been standardized.
- Reshape annual and monthly datasets into analysis-ready formats.
- Create clean analytical tables for further analysis.


In [1]:
import pandas as pd
import numpy as np

## 1. Load Raw Datasets

The datasets are loaded from the `data/raw/` directory. The original raw files are kept unchanged so that the cleaning process remains reproducible.


In [2]:
cpi = pd.read_csv("../data/raw/CPI_CMA_2006_2024.csv", encoding="latin1")
crime = pd.read_csv("../data/raw/crime_rate_CMA_2006_2023.csv", encoding="latin1")
housing = pd.read_csv("../data/raw/housing_price_index_by_CMA_2006_2023.csv", encoding="latin1")
population = pd.read_csv("../data/raw/population_by_CMA_2006_2023.csv", encoding="latin1")
unemployment = pd.read_csv("../data/raw/unemployment_by_CMA_2006_2024.csv", encoding="latin1")

datasets = {
    "CPI": cpi,
    "Crime": crime,
    "Housing": housing,
    "Population": population,
    "Unemployment": unemployment
}

In [3]:
print(crime.dtypes)
print(population.dtypes)

City    object
2006    object
2007    object
2008    object
2009    object
2010    object
2011    object
2012    object
2013    object
2014    object
2015    object
2016    object
2017    object
2018    object
2019    object
2020    object
2021    object
2022    object
2023    object
dtype: object
City    object
2006    object
2007    object
2008    object
2009    object
2010    object
2011    object
2012    object
2013    object
2014    object
2015    object
2016    object
2017    object
2018    object
2019    object
2020    object
2021    object
2022    object
2023    object
2024    object
dtype: object


## 2. Standardize City Names

The city names should be consistent across all datasets before combining or comparing the data.

A naming inconsistency was identified in the Housing dataset, where Montréal is written as `Montréal, Quebec` instead of `Montréal`.

We will standardize the city names while preserving the original city labels as much as possible.

In [4]:
for name, df in datasets.items():
    print(f"{name}:")
    print(df["City"].unique())
    print()

CPI:
['Montréal' 'Ottawa-Gatineau' 'Toronto' 'Winnipeg' 'Regina' 'Saskatoon'
 'Edmonton' 'Calgary' 'Vancouver']

Crime:
['Montréal' 'Ottawa-Gatineau' 'Toronto' 'Winnipeg' 'Regina' 'Saskatoon'
 'Calgary' 'Edmonton' 'Vancouver']

Housing:
['Montréal, Quebec' 'Ottawa-Gatineau' 'Toronto' 'Winnipeg' 'Regina'
 'Saskatoon' 'Calgary' 'Edmonton' 'Vancouver']

Population:
['Montréal' 'Ottawa-Gatineau' 'Toronto' 'Winnipeg' 'Regina' 'Saskatoon'
 'Calgary' 'Edmonton' 'Vancouver']

Unemployment:
['Montréal' 'Ottawa-Gatineau' 'Toronto' 'Winnipeg' 'Regina' 'Saskatoon'
 'Calgary' 'Edmonton' 'Vancouver']



The city names are consistent across five datasets. The only inconsistency is in the Housing dataset, where Montréal is written as `Montréal, Quebec` instead of `Montréal`.

This inconsistency will be corrected so that the same city identifier can be used consistently across all datasets.


In [5]:
housing["City"] = housing["City"].replace("Montréal, Quebec", "Montréal")

housing["City"].unique()

array(['Montréal', 'Ottawa-Gatineau', 'Toronto', 'Winnipeg', 'Regina',
       'Saskatoon', 'Calgary', 'Edmonton', 'Vancouver'], dtype=object)

The city names are now standardized across all datasets. All five datasets use the same nine city names, which will allow them to be compared and combined consistently during the analysis.

## 3. Inspect Data Types

The previous inspection showed that some numerical variables are stored as `object` rather than numeric data types.

We will inspect the current data types of each dataset before applying any conversions. This will help us identify which columns require cleaning.

In [6]:
for name, df in datasets.items():
    print(f"{name}:")
    print(df.dtypes.value_counts())
    print()

CPI:
float64    234
object       1
Name: count, dtype: int64

Crime:
object    19
Name: count, dtype: int64

Housing:
float64    232
object       1
int64        1
Name: count, dtype: int64

Population:
object    20
Name: count, dtype: int64

Unemployment:
float64    19
object      1
Name: count, dtype: int64



The data types are not fully consistent across the datasets. CPI and Unemployment are already mostly stored as numeric values, while Crime and Population contain numeric values stored as `object` types. The Housing dataset also contains a small number of columns with different data types.

These inconsistencies need to be investigated and corrected before performing numerical analysis.

## 5. Inspect Object-Type Values

Before converting the object-type columns to numeric values, we will inspect their contents.

This allows us to identify formatting issues such as commas, spaces, or other non-numeric characters that may need to be cleaned first.


In [7]:
for name in ["Crime", "Population"]:
    print(f"\n {name} : \n")
    print(datasets[name].head())


 Crime : 

              City       2006       2007       2008       2009       2010  \
0         Montréal   6,529.55   6,170.03   6,201.21   6,076.36   5,644.45   
1  Ottawa-Gatineau   6,141.23   5,994.33   5,494.68   5,296.26   5,004.38   
2          Toronto   4,971.41   4,731.08   4,493.42   4,284.50   4,060.66   
3         Winnipeg  11,343.97   9,954.29   8,587.67   9,555.69   7,904.73   
4           Regina  14,261.82  13,637.43  13,159.08  11,982.80  11,318.79   

        2011       2012      2013      2014      2015       2016      2017  \
0   5,402.13   5,193.47  4,690.55  4,321.46  4,242.47   3,999.89  3,987.80   
1   4,785.08   4,775.49  4,203.55  4,035.33  3,982.63   4,051.58  4,063.30   
2   3,889.87   3,631.85  3,365.65  3,273.82  3,320.70   3,398.56  3,544.03   
3   6,896.09   6,687.89  5,751.00  5,758.84  6,260.38   6,995.46  7,323.62   
4  10,957.50  10,184.81  9,281.03  8,890.58  9,234.93  10,278.80  9,694.26   

        2018       2019      2020      2021      2022   

In [8]:
for name in ["Crime", "Population"]:
    print(name)
    print(datasets[name].dtypes)
    print()

Crime
City    object
2006    object
2007    object
2008    object
2009    object
2010    object
2011    object
2012    object
2013    object
2014    object
2015    object
2016    object
2017    object
2018    object
2019    object
2020    object
2021    object
2022    object
2023    object
dtype: object

Population
City    object
2006    object
2007    object
2008    object
2009    object
2010    object
2011    object
2012    object
2013    object
2014    object
2015    object
2016    object
2017    object
2018    object
2019    object
2020    object
2021    object
2022    object
2023    object
2024    object
dtype: object



The yearly columns in the `Population` and `Crime` datasets are stored as `object` types, even though they contain numerical values.

The next step is to convert these columns to appropriate numeric types.

## 6. Convert Numeric Columns to Appropriate Data Types

The Crime and Population datasets contain numerical values stored as text.

The values use commas as thousands separators, so the commas must be removed before converting the columns to numeric types.

### 6.1 Convert Crime Data

In [9]:
crime_numeric_cols = crime.columns.drop("City")

crime[crime_numeric_cols] = (
    crime[crime_numeric_cols]
    .replace(",", "", regex=True)
    .astype(float)
)

crime.dtypes

City     object
2006    float64
2007    float64
2008    float64
2009    float64
2010    float64
2011    float64
2012    float64
2013    float64
2014    float64
2015    float64
2016    float64
2017    float64
2018    float64
2019    float64
2020    float64
2021    float64
2022    float64
2023    float64
dtype: object

The Crime dataset has been successfully converted to numeric data types. All yearly crime rate columns from 2006 to 2023 are now stored as `float64`, while the `City` column remains an object type as expected.

### 6.3 Convert Population Data

In [10]:
population_numeric_cols = population.columns.drop("City")

population[population_numeric_cols] = (
    population[population_numeric_cols]
    .replace(",", "", regex=True)
    .astype(float)
)

population.dtypes

City     object
2006    float64
2007    float64
2008    float64
2009    float64
2010    float64
2011    float64
2012    float64
2013    float64
2014    float64
2015    float64
2016    float64
2017    float64
2018    float64
2019    float64
2020    float64
2021    float64
2022    float64
2023    float64
2024    float64
dtype: object

The Population dataset has been successfully converted to numeric data types. All yearly values from 2006 to 2024 are now stored as `float64`, while the `City` column remains an object type as expected.

### 6.4 Review Data Types After Conversion

In [11]:
for name, df in datasets.items():
    print(f"{name}:")
    print(df.dtypes.value_counts())
    print()

CPI:
float64    234
object       1
Name: count, dtype: int64

Crime:
float64    18
object      1
Name: count, dtype: int64

Housing:
float64    232
object       1
int64        1
Name: count, dtype: int64

Population:
float64    19
object      1
Name: count, dtype: int64

Unemployment:
float64    19
object      1
Name: count, dtype: int64



The data types are now appropriate for numerical analysis. All temporal variables are stored as numeric types, while the `City` column remains an object type.

The Housing dataset contains both `float64` and `int64` numerical columns, which is not a problem because both types can be used for numerical analysis.

## 7. Reshape Annual Datasets

### 7.1 Reshape Crime Data

In [12]:
crime_long = crime.melt(
    id_vars="City",
    var_name="Year",
    value_name="Crime_Rate"
)

#crime_long.head()
crime.head()

,City,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
0,Montréal,6529.55,6170.03,6201.21,6076.36,5644.45,5402.13,5193.47,4690.55,4321.46,4242.47,3999.89,3987.80,3835.02,3844.37,3484.55,3731.79,4126.31,4320.08
1,Ottawa-Gatineau,6141.23,5994.33,5494.68,5296.26,5004.38,4785.08,4775.49,4203.55,4035.33,3982.63,4051.58,4063.30,4241.99,4374.69,3517.03,3626.80,4160.89,4406.34
2,Toronto,4971.41,4731.08,4493.42,4284.50,4060.66,3889.87,3631.85,3365.65,3273.82,3320.70,3398.56,3544.03,3772.19,3726.29,3176.27,3294.62,3832.16,4391.64
3,Winnipeg,11343.97,9954.29,8587.67,9555.69,7904.73,6896.09,6687.89,5751.00,5758.84,6260.38,6995.46,7323.62,8121.80,8873.01,7850.56,7279.92,8930.20,8362.96
4,Regina,14261.82,13637.43,13159.08,11982.80,11318.79,10957.50,10184.81,9281.03,8890.58,9234.93,10278.80,9694.26,10430.41,10644.14,7947.95,8267.95,8226.79,8418.51


The Crime dataset has been successfully reshaped from wide format to long format. Each row now represents one city and one year, with the corresponding crime rate stored in the `Crime_Rate` column.

This structure will make it easier to perform time-based analysis, compare cities, and integrate the data with the other annual datasets.

### 7.3 Reshape Population Data

In [13]:
population_long = population.melt(
    id_vars="City",
    var_name="Year",
    value_name="Population"
)

population_long.head()

,City,Year,Population
0,Montréal,2006,3088.5
1,Ottawa-Gatineau,2006,960.9
2,Toronto,2006,4322.1
3,Winnipeg,2006,579.5
4,Regina,2006,162.8


The Population dataset has been successfully reshaped from wide format to long format. Each row now represents one city and one year, with the corresponding population value stored in the `Population` column.

This structure will make it easier to analyze population trends over time and compare cities consistently with the other annual indicators.

### 7.4 Reshape Unemployment Data

In [14]:
unemployment_long = unemployment.melt(
    id_vars="City",
    var_name="Year",
    value_name="Unemployment_Rate"
)

unemployment_long.head()

,City,Year,Unemployment_Rate
0,Montréal,2006,8.6
1,Ottawa-Gatineau,2006,5.4
2,Toronto,2006,6.7
3,Winnipeg,2006,4.9
4,Regina,2006,4.9


The Unemployment dataset has been successfully reshaped from wide format to long format. Each row now represents one city and one year, with the corresponding unemployment rate stored in the `Unemployment_Rate` column.

This structure will make it easier to analyze unemployment trends over time and compare cities consistently with the other annual indicators.

In [15]:
annual_datasets = {
    "Crime": crime_long,
    "Population": population_long,
    "Unemployment": unemployment_long
}

for name, df in annual_datasets.items():
    print(f"{name}:")
    print(df["Year"].dtype)
    print()


Crime:
object

Population:
object

Unemployment:
object



The annual datasets contain the expected number of observations for each city and year.

Crime contains 162 observations, covering 2006 to 2023, while Population, and Unemployment each contain 171 observations, covering 2006 to 2024.

The `Year` column is currently stored as a string. It will be converted to an integer so that it can be used correctly in numerical and time-based analysis.

### 7.6 Convert Year to Integer

In [16]:
for name, df in annual_datasets.items():
    df["Year"] = df["Year"].astype(int)

for name, df in annual_datasets.items():
    print(f"{name}:")
    print(df["Year"].dtype)
    print()

Crime:
int64

Population:
int64

Unemployment:
int64



The `Year` column has been successfully converted from string to integer in all four annual datasets.

The datasets are now ready for further transformation and integration.

### 7.7 Inspect Monthly CPI Columns

In [32]:
cpi.columns[:10].tolist(), cpi.columns[-10:].tolist()

(['City',
  '06-Jan',
  '06-Feb',
  '06-Mar',
  '06-Apr',
  '06-May',
  '06-Jun',
  '06-Jul',
  '06-Aug',
  '06-Sep'],
 ['24-Sep',
  '24-Oct',
  '24-Nov',
  '24-Dec',
  '25-Jan',
  '25-Feb',
  '25-Mar',
  '25-Apr',
  '25-May',
  '25-Jun'])

The CPI dataset contains monthly observations from January 2006 to June 2025.

The monthly columns follow a consistent `YY-Mon` format, such as `06-Jan` for January 2006 and `25-Jun` for June 2025.

The dataset is ready to be reshaped from wide format to long format.

### 7.8 Reshape CPI Data

In [34]:
cpi_long = cpi.melt(
    id_vars="City",
    var_name="Month",
    value_name="CPI"
)

cpi_long.head()

,City,Month,CPI
0,Montréal,06-Jan,107.9
1,Ottawa-Gatineau,06-Jan,108.1
2,Toronto,06-Jan,107.9
3,Winnipeg,06-Jan,107.2
4,Regina,06-Jan,107.7


The CPI dataset has been successfully reshaped from wide format to long format. Each row now represents one city and one month, with the corresponding CPI value stored in the `CPI` column.

This structure will make it easier to analyze monthly CPI trends and integrate the data with other datasets when required.

### 7.9 Create CPI Date Column

In [35]:
cpi_long["Date"] = pd.to_datetime(
    cpi_long["Month"],
    format="%y-%b"
)

cpi_long.head()

,City,Month,CPI,Date
0,Montréal,06-Jan,107.9,2006-01-01
1,Ottawa-Gatineau,06-Jan,108.1,2006-01-01
2,Toronto,06-Jan,107.9,2006-01-01
3,Winnipeg,06-Jan,107.2,2006-01-01
4,Regina,06-Jan,107.7,2006-01-01


The `Date` column has been successfully created as a datetime variable. The original monthly labels are preserved in the `Month` column, while the new `Date` column provides a standardized date format for time-based analysis.

The CPI dataset is now ready for further transformation.

### 7.10 Inspect Monthly Housing Columns

In [20]:
housing.columns[:10].tolist(), housing.columns[-10:].tolist()

(['City',
  '06-Jan',
  '06-Feb',
  '06-Mar',
  '06-Apr',
  '06-May',
  '06-Jun',
  '06-Jul',
  '06-Aug',
  '06-Sep'],
 ['24-Aug',
  '24-Sep',
  '24-Oct',
  '24-Nov',
  '24-Dec',
  '25-Jan',
  '25-Feb',
  '25-Mar',
  '25-Apr',
  '25-May'])

The Housing dataset contains monthly observations from January 2006 to May 2025.

The monthly columns follow the same `YY-Mon` format as the CPI dataset. This consistent structure allows the Housing data to be transformed using the same approach.

### 7.11 Reshape Housing Data

In [21]:
housing_long = housing.melt(
    id_vars="City",
    var_name="Month",
    value_name="Housing_Price_Index"
)

housing_long.head()

,City,Month,Housing_Price_Index
0,Montréal,06-Jan,78.7
1,Ottawa-Gatineau,06-Jan,84.2
2,Toronto,06-Jan,69.1
3,Winnipeg,06-Jan,59.9
4,Regina,06-Jan,50.6


The Housing dataset has been successfully reshaped from wide format to long format. Each row now represents one city and one month, with the corresponding housing price index stored in the `Housing_Price_Index` column.

This structure will make it easier to analyze housing price trends over time and compare cities consistently.

### 7.12 Create Housing Date Column

In [22]:
housing_long["Date"] = pd.to_datetime(
    housing_long["Month"],
    format="%y-%b"
)

housing_long.head()

,City,Month,Housing_Price_Index,Date
0,Montréal,06-Jan,78.7,2006-01-01
1,Ottawa-Gatineau,06-Jan,84.2,2006-01-01
2,Toronto,06-Jan,69.1,2006-01-01
3,Winnipeg,06-Jan,59.9,2006-01-01
4,Regina,06-Jan,50.6,2006-01-01


The `Date` column has been successfully created as a datetime variable. The original monthly labels are preserved in the `Month` column, while the new `Date` column provides a standardized format for time-based analysis.

The Housing dataset is now ready for further validation and integration.

### 7.13 Validate Monthly Data

In [23]:
monthly_datasets = {
    "CPI": cpi_long,
    "Housing": housing_long
}

for name, df in monthly_datasets.items():
    print(f"{name}:")
    print("Number of rows:", len(df))
    print("Number of cities:", df["City"].nunique())
    print("Date range:", df["Date"].min(), "to", df["Date"].max())
    print()

CPI:
Number of rows: 2106
Number of cities: 9
Date range: 2006-01-01 00:00:00 to 2025-06-01 00:00:00

Housing:
Number of rows: 2097
Number of cities: 9
Date range: 2006-01-01 00:00:00 to 2025-05-01 00:00:00



The monthly datasets have been successfully reshaped and validated.

The CPI dataset contains 2,106 observations, corresponding to 9 cities across 234 months from January 2006 to June 2025.

The Housing dataset contains 2,097 observations, corresponding to 9 cities across 233 months from January 2006 to May 2025.

The datasets have the same nine cities and a consistent monthly structure. The one-month difference in temporal coverage will be considered when combining the datasets.

## 8. Validate Cleaned Datasets

### 8.1 Check Missing Values

In [24]:
cleaned_datasets = {
    "Crime": crime_long,
    "Population": population_long,
    "Unemployment": unemployment_long,
    "CPI": cpi_long,
    "Housing": housing_long
}

for name, df in cleaned_datasets.items():
    print(f"{name}:")
    print(df.isna().sum())
    print()

Crime:
City          0
Year          0
Crime_Rate    0
dtype: int64

Population:
City          0
Year          0
Population    0
dtype: int64

Unemployment:
City                 0
Year                 0
Unemployment_Rate    0
dtype: int64

CPI:
City     0
Month    0
CPI      0
Date     0
dtype: int64

Housing:
City                   0
Month                  0
Housing_Price_Index    0
Date                   0
dtype: int64



No missing values were introduced during the cleaning and reshaping process.

All five cleaned datasets contain complete values for their respective columns, so no imputation or removal of observations is required at this stage.

### 8.2 Check Duplicate Records

In [25]:
for name, df in cleaned_datasets.items():
    print(f"{name}:")
    print("Duplicate rows:", df.duplicated().sum())
    print()

Crime:
Duplicate rows: 0

Population:
Duplicate rows: 0

Unemployment:
Duplicate rows: 0

CPI:
Duplicate rows: 0

Housing:
Duplicate rows: 0



No duplicate rows were found in any of the five cleaned datasets.

Each observation therefore represents a unique record at the current level of granularity.

### 8.3 Validate Final Data Types

In [26]:
for name, df in cleaned_datasets.items():
    print(f"{name}:")
    print(df.dtypes)
    print()

Crime:
City           object
Year            int64
Crime_Rate    float64
dtype: object

Population:
City           object
Year            int64
Population    float64
dtype: object

Unemployment:
City                  object
Year                   int64
Unemployment_Rate    float64
dtype: object

CPI:
City             object
Month            object
CPI             float64
Date     datetime64[ns]
dtype: object

Housing:
City                           object
Month                          object
Housing_Price_Index           float64
Date                   datetime64[ns]
dtype: object



The final data types are appropriate for analysis.

The annual datasets contain integer `Year` values and numerical indicator columns stored as `float64`. The monthly datasets contain numerical indicator columns stored as `float64` and standardized `Date` columns stored as `datetime64[ns]`.

The `City` and original `Month` labels remain as object types, which is appropriate for categorical and textual information.

The cleaned datasets are now ready for the next transformation stage.

## 9. Export Processed Datasets

The cleaned datasets are now ready to be exported.

Saving the processed datasets separately will allow us to reuse them later for SQL analysis, machine learning, and Power BI without modifying the original raw files.

In [27]:
import os

os.makedirs("../data/processed", exist_ok=True)

crime_long.to_csv("../data/processed/crime_clean.csv", index=False)
population_long.to_csv("../data/processed/population_clean.csv", index=False)
unemployment_long.to_csv("../data/processed/unemployment_clean.csv", index=False)
cpi_long.to_csv("../data/processed/cpi_clean.csv", index=False)
housing_long.to_csv("../data/processed/housing_clean.csv", index=False)

print("All processed datasets have been exported successfully.")

All processed datasets have been exported successfully.


All five processed datasets have been successfully exported and validated.

The exported files exist in the `data/processed/` directory, and their row counts match the cleaned datasets created in this notebook.

The processed datasets are now ready for exploratory data analysis, SQL analysis, and machine learning.